In [1]:
import os
import joblib
import pandas as pd
import numpy as np

#### Constants

In [2]:
str_bucket = os.getcwd().split('/')[4].replace('_','-')
print(f'Bucket: {str_bucket}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

str_target = 'pitch_type'

str_dirname_output = './output'
os.makedirs(str_dirname_output, exist_ok=True)

# most common pitch types
list_str_pitch_type = [
    'FF',
    'SL',
    'SI',
    'FT',
    'CH',
    'CU',
    'FC',
    'FS',
]

Bucket: assessment-swish-analytics
Task: 03_preprocessing


#### Transformer Classes

##### SituationFeatures

Creates game-situation features derived from columns that are known prior to the pitch being thrown. This transformer is stateless (fit is a no-op).

- **score_diff**: Score differential from the pitching team's perspective. When `top == 1` the pitching team is the home team, so `home_team_runs - away_team_runs`; otherwise the reverse.
- **on_1b_flag / on_2b_flag / on_3b_flag**: Binary indicators for whether a runner occupies first, second, or third base, derived from the non-null status of `on_1b`, `on_2b`, and `on_3b`.
- **runners_on**: Total number of runners on base (sum of the three base flags).
- **bases_loaded**: Binary indicator for whether all three bases are occupied (`runners_on == 3`). Bases-loaded situations are high-leverage and influence pitch selection — pitchers often favor ground-ball pitches (sinkers, two-seamers) to induce double plays.
- **same_hand**: Binary indicator for whether the pitcher and batter share the same handedness (`p_throws == stand`), capturing the platoon advantage.
- **is_first_pitch**: Binary indicator for whether this is the first pitch of the at-bat (`pcount_at_bat == 1`), since pitch selection tendencies differ significantly on the first pitch.
- **b_height_inches**: Batter height converted from string format (e.g., "6-2") to total inches (e.g., 74). Taller batters have larger strike zones, which influences pitch selection — pitchers may throw more high fastballs to tall batters and more low breaking balls to shorter batters.

In [ ]:
class SituationFeatures:
    def __init__(self):
        pass
    def fit(self, X):
        print('Situation features...')
        return self
    def transform(self, X):
        print('Situation features...')
        X = X.copy()
        # score differential from pitching team perspective
        X['score_diff'] = np.where(
            X['top'] == 1,
            X['home_team_runs'] - X['away_team_runs'],
            X['away_team_runs'] - X['home_team_runs']
        )
        # runners on base
        X['on_1b_flag'] = X['on_1b'].notna().astype(int)
        X['on_2b_flag'] = X['on_2b'].notna().astype(int)
        X['on_3b_flag'] = X['on_3b'].notna().astype(int)
        X['runners_on'] = X['on_1b_flag'] + X['on_2b_flag'] + X['on_3b_flag']
        X['bases_loaded'] = (X['runners_on'] == 3).astype(int)
        # handedness matchup
        X['same_hand'] = (X['p_throws'] == X['stand']).astype(int)
        # is first pitch of at-bat
        X['is_first_pitch'] = (X['pcount_at_bat'] == 1).astype(int)
        # batter height (convert string like "6-2" to inches)
        X['b_height_inches'] = X['b_height'].apply(
            lambda x: int(str(x).split('-')[0]) * 12 + int(str(x).split('-')[1]) if pd.notna(x) and '-' in str(x) else np.nan
        )
        return X

##### LagFeatures

Creates within-at-bat lag features that capture pitch sequencing patterns. This transformer is stateless (fit is a no-op).

The data is first sorted by `game_pk`, `at_bat_num`, and `pcount_at_bat` to ensure correct chronological ordering within each at-bat before computing lags.

- **prev_pitch_type**: The pitch type thrown on the immediately preceding pitch within the same at-bat, obtained via a grouped shift(1). Filled with `'NONE'` for the first pitch of each at-bat.
- **prev_pitch_type_2**: The pitch type thrown two pitches prior within the same at-bat, obtained via a grouped shift(2). Filled with `'NONE'` for the first and second pitches of each at-bat.

These features allow the model to learn sequential dependencies in pitch selection — for example, a pitcher who just threw a fastball may be more likely to follow with an off-speed pitch.

In [4]:
class LagFeatures:
    def __init__(self):
        pass
    def fit(self, X):
        print('Lag features...')
        return self
    def transform(self, X):
        print('Lag features...')
        X = X.copy()
        X = X.sort_values(['game_pk', 'at_bat_num', 'pcount_at_bat'])
        # previous pitch type in this at-bat
        X['prev_pitch_type'] = X.groupby(['game_pk', 'at_bat_num'])['pitch_type'].shift(1)
        X['prev_pitch_type'] = X['prev_pitch_type'].fillna('NONE')
        # 2nd previous pitch type in this at-bat
        X['prev_pitch_type_2'] = X.groupby(['game_pk', 'at_bat_num'])['pitch_type'].shift(2)
        X['prev_pitch_type_2'] = X['prev_pitch_type_2'].fillna('NONE')
        return X

##### GameContextFeatures

Creates game-context features that capture the broader game state beyond the immediate pitch situation. This transformer is stateless (fit is a no-op).

- **total_runs**: Total runs scored in the game so far (`away_team_runs + home_team_runs`). High-scoring games may have different pitch selection patterns — pitchers may be more aggressive or conservative depending on run environment.
- **close_game**: Binary indicator for whether the game is close (`|score_diff| <= 2`). In close games, pitchers tend to pitch more carefully, favoring higher-percentage pitches.
- **late_inning**: Binary indicator for innings 7 or later. Late-game situations (especially with a close score) carry higher leverage, influencing pitch selection toward safer, higher-confidence choices.

In [5]:
class GameContextFeatures:
    def __init__(self):
        pass
    def fit(self, X):
        print('Game context features...')
        return self
    def transform(self, X):
        print('Game context features...')
        X = X.copy()
        # total runs scored in game so far
        X['total_runs'] = X['away_team_runs'] + X['home_team_runs']
        # close game indicator
        X['close_game'] = (X['score_diff'].abs() <= 2).astype(int)
        # late inning indicator
        X['late_inning'] = (X['inning'] >= 7).astype(int)
        return X

##### PitcherStats

Computes pitcher-level aggregate statistics that characterize each pitcher's tendencies. This is a stateful transformer — fit learns the statistics from training data only (to prevent data leakage), and transform merges them onto any split.

**Learned during fit:**
- **df_pitcher_mix**: Each pitcher's overall pitch type distribution (proportion of each pitch type thrown), computed via `pd.crosstab` with row normalization. Produces columns like `pitcher_pct_FF`, `pitcher_pct_SL`, etc.
- **df_pitcher_count**: Each pitcher's pitch type distribution conditional on the ball-strike count (e.g., what a pitcher throws in a 0-2 count vs. a 3-1 count). Produces columns like `pitcher_count_pct_FF`, `pitcher_count_pct_SL`, etc.
- **sr_pitcher_n_types**: Number of distinct pitch types each pitcher throws (`pitcher_n_types`), capturing repertoire diversity.

**Applied during transform:**
All three sets of statistics are left-joined onto the input dataframe via `pitcher_id` (and `count_str` for the count-conditional mix). Pitchers not seen during training will have NaN values for these features, which are handled downstream by `PrepareFeatures` (filled with 0).

In [ ]:
class PitcherStats:
    def __init__(self):
        pass
    def fit(self, X):
        print('Computing pitcher stats...')
        # overall pitch type distribution per pitcher
        df_pitcher_mix = pd.crosstab(X['pitcher_id'], X['pitch_type'], normalize='index')
        df_pitcher_mix.columns = [f'pitcher_pct_{c}' for c in df_pitcher_mix.columns]
        self.df_pitcher_mix = df_pitcher_mix
        # pitch type distribution per pitcher per count
        X_tmp = X.copy()
        X_tmp['count_str'] = X_tmp['balls'].astype(str) + '-' + X_tmp['strikes'].astype(str)
        df_pitcher_count = pd.crosstab(
            [X_tmp['pitcher_id'], X_tmp['count_str']],
            X_tmp['pitch_type'],
            normalize='index'
        )
        df_pitcher_count.columns = [f'pitcher_count_pct_{c}' for c in df_pitcher_count.columns]
        self.df_pitcher_count = df_pitcher_count
        # pitcher repertoire size
        self.sr_pitcher_n_types = X.groupby('pitcher_id')['pitch_type'].nunique().rename('pitcher_n_types')
        return self
    def transform(self, X):
        print('Applying pitcher stats...')
        X = X.copy()
        X['count_str'] = X['balls'].astype(str) + '-' + X['strikes'].astype(str)
        # overall pitcher mix
        X = X.merge(self.df_pitcher_mix, left_on='pitcher_id', right_index=True, how='left')
        # pitcher mix per count
        X = X.merge(self.df_pitcher_count, left_on=['pitcher_id', 'count_str'], right_index=True, how='left')
        # pitcher repertoire
        X = X.merge(self.sr_pitcher_n_types, left_on='pitcher_id', right_index=True, how='left')
        return X

##### PitcherHandednessStats

Computes pitcher pitch mix statistics conditioned on batter handedness matchup. This is a stateful transformer — fit learns the statistics from training data only.

Pitchers throw very different pitch mixes depending on whether they face a same-hand or opposite-hand batter. For example, a left-handed pitcher might throw 40% sliders to right-handed batters but only 15% to left-handed batters. This transformer captures that interaction by computing separate pitch type distributions per `(pitcher_id, same_hand)` combination.

**Learned during fit:**
- **df_pitcher_hand_mix**: Each pitcher's pitch type distribution conditioned on `same_hand` (0 or 1), computed via `pd.crosstab` with row normalization. Produces columns like `pitcher_hand_pct_FF`, `pitcher_hand_pct_SL`, etc.

**Applied during transform:**
The pitcher-handedness mix is left-joined onto the input dataframe via `(pitcher_id, same_hand)`. Pitchers or matchups not seen during training will have NaN values, handled downstream by `PrepareFeatures` (filled with 0).

In [7]:
class PitcherHandednessStats:
    def __init__(self):
        pass
    def fit(self, X):
        print('Computing pitcher handedness stats...')
        df_pitcher_hand_mix = pd.crosstab(
            [X['pitcher_id'], X['same_hand']],
            X['pitch_type'],
            normalize='index'
        )
        df_pitcher_hand_mix.columns = [f'pitcher_hand_pct_{c}' for c in df_pitcher_hand_mix.columns]
        self.df_pitcher_hand_mix = df_pitcher_hand_mix
        return self
    def transform(self, X):
        print('Applying pitcher handedness stats...')
        X = X.copy()
        X = X.merge(
            self.df_pitcher_hand_mix,
            left_on=['pitcher_id', 'same_hand'],
            right_index=True,
            how='left'
        )
        return X

##### PitcherHandednessCountStats

Computes pitcher pitch mix statistics conditioned on both batter handedness matchup AND ball-strike count. This is a stateful transformer — fit learns the statistics from training data only.

This captures the most granular interaction between pitcher identity, matchup, and count. For example, a right-handed pitcher facing a left-handed batter in a 0-2 count might throw 50% sliders, while the same pitcher in a 3-1 count to a right-handed batter might throw 80% fastballs. These three-way interaction patterns are difficult for tree models to learn from raw features alone.

**Learned during fit:**
- **df_mix**: Each pitcher's pitch type distribution conditioned on `(same_hand, count_str)`, computed via `pd.crosstab` with row normalization. Produces columns like `pitcher_hand_count_pct_FF`, `pitcher_hand_count_pct_SL`, etc.

**Applied during transform:**
The statistics are left-joined onto the input dataframe via `(pitcher_id, same_hand, count_str)`. Combinations not seen during training will have NaN values, handled downstream by `PrepareFeatures` (filled with 0).

In [8]:
class PitcherHandednessCountStats:
    def __init__(self):
        pass
    def fit(self, X):
        print('Computing pitcher handedness count stats...')
        X_tmp = X.copy()
        X_tmp['count_str'] = X_tmp['balls'].astype(str) + '-' + X_tmp['strikes'].astype(str)
        df_mix = pd.crosstab(
            [X_tmp['pitcher_id'], X_tmp['same_hand'], X_tmp['count_str']],
            X_tmp['pitch_type'],
            normalize='index'
        )
        df_mix.columns = [f'pitcher_hand_count_pct_{c}' for c in df_mix.columns]
        self.df_mix = df_mix
        return self
    def transform(self, X):
        print('Applying pitcher handedness count stats...')
        X = X.copy()
        X['count_str'] = X['balls'].astype(str) + '-' + X['strikes'].astype(str)
        X = X.merge(
            self.df_mix,
            left_on=['pitcher_id', 'same_hand', 'count_str'],
            right_index=True,
            how='left'
        )
        return X

##### BatterStats

Computes batter-level aggregate statistics that characterize the pitch mix each batter typically faces. This is a stateful transformer — fit learns the statistics from training data only (to prevent data leakage), and transform merges them onto any split.

**Learned during fit:**
- **df_batter_mix**: The distribution of pitch types each batter has faced in training, computed via `pd.crosstab` with row normalization. Produces columns like `batter_pct_FF`, `batter_pct_SL`, etc. This captures that some batters see disproportionately more fastballs (e.g., power hitters) or breaking balls (e.g., batters who struggle with off-speed).

**Applied during transform:**
The batter mix statistics are left-joined onto the input dataframe via `batter_id`. Batters not seen during training will have NaN values, handled downstream by `PrepareFeatures` (filled with 0).

In [ ]:
class BatterStats:
    def __init__(self):
        pass
    def fit(self, X):
        print('Computing batter stats...')
        # pitch type distribution faced by each batter
        df_batter_mix = pd.crosstab(X['batter_id'], X['pitch_type'], normalize='index')
        df_batter_mix.columns = [f'batter_pct_{c}' for c in df_batter_mix.columns]
        self.df_batter_mix = df_batter_mix
        return self
    def transform(self, X):
        print('Applying batter stats...')
        X = X.copy()
        X = X.merge(self.df_batter_mix, left_on='batter_id', right_index=True, how='left')
        return X

##### CountCategory

Creates a simplified categorical representation of the ball-strike count from the pitcher's perspective. This transformer is stateless (fit is a no-op).

While the raw `balls` and `strikes` columns are already included as numeric features, tree-based models must learn count-based strategies through multiple splits on two separate columns. This transformer provides a single categorical feature that captures the strategic state of the count directly:

- **first_pitch**: 0 balls, 0 strikes — pitchers often start with a fastball for a strike.
- **ahead**: strikes > balls — pitcher has the advantage, can throw more breaking/off-speed.
- **behind**: balls > strikes — pitcher is behind, more likely to throw fastballs for strikes.
- **even**: balls == strikes (but not 0-0) — neutral count.
- **full_count**: 3 balls, 2 strikes — high-pressure count with distinct tendencies.

In [10]:
class CountCategory:
    def __init__(self):
        pass
    def fit(self, X):
        print('Count category...')
        return self
    def transform(self, X):
        print('Count category...')
        X = X.copy()
        conditions = [
            (X['balls'] == 0) & (X['strikes'] == 0),
            (X['balls'] == 3) & (X['strikes'] == 2),
            X['strikes'] > X['balls'],
            X['balls'] > X['strikes'],
        ]
        choices = ['first_pitch', 'full_count', 'ahead', 'behind']
        X['count_category'] = np.select(conditions, choices, default='even')
        return X

##### PrepareFeatures

Selects, encodes, and aligns the final feature matrix for modeling. This is a stateful transformer — fit learns the complete set of column names from the training data so that valid/test splits are aligned to the same feature space.

**Learned during fit:**
- **list_pitcher_mix / list_pitcher_count_mix / list_pitcher_hand_mix / list_pitcher_hand_count_mix / list_batter_mix**: Dynamically detected column names for all mix-type features (any column starting with `pitcher_pct_`, `pitcher_count_pct_`, `pitcher_hand_pct_`, `pitcher_hand_count_pct_`, or `batter_pct_`).
- **list_all_cols**: The sorted, complete list of all feature column names after combining numeric, mix, and one-hot encoded categorical features on the training set. This serves as the canonical column order for all splits.

**Applied during transform:**
1. **Numeric features**: Selects the pre-defined `list_numeric` columns (e.g., `inning`, `outs`, `balls`, `strikes`, `score_diff`, `total_runs`, `close_game`, `late_inning`, runner flags, `at_bat_num`, `b_height_inches`, `pitcher_n_types`).
2. **Mix features**: Selects all pitcher mix, pitcher-count mix, pitcher-handedness mix, pitcher-handedness-count mix, and batter mix columns, filling NaN with 0 for unseen pitchers/batters.
3. **Categorical features**: One-hot encodes `p_throws`, `stand`, `prev_pitch_type`, `prev_pitch_type_2`, and `count_category` using `pd.get_dummies`.
4. **Column alignment**: Reindexes the combined feature matrix to `list_all_cols` with `fill_value=0`, ensuring valid/test have exactly the same columns as training.

Note: The target variable (`y`) is extracted separately outside this transformer since `PreprocessingModel.transform` returns only the feature matrix `X`.

In [11]:
class PrepareFeatures:
    def __init__(self, list_numeric, list_categorical):
        self.list_numeric = list_numeric
        self.list_categorical = list_categorical
    def fit(self, X):
        print('Preparing features...')
        # learn pitcher mix and count mix column names from training data
        self.list_pitcher_mix = [c for c in X.columns if c.startswith('pitcher_pct_')]
        self.list_pitcher_count_mix = [c for c in X.columns if c.startswith('pitcher_count_pct_')]
        self.list_pitcher_hand_mix = [c for c in X.columns if c.startswith('pitcher_hand_pct_')]
        self.list_pitcher_hand_count_mix = [c for c in X.columns if c.startswith('pitcher_hand_count_pct_')]
        self.list_batter_mix = [c for c in X.columns if c.startswith('batter_pct_')]
        # learn aligned column list from training data
        X_num = X[self.list_numeric].copy()
        list_mix_cols = (self.list_pitcher_mix + self.list_pitcher_count_mix +
                         self.list_pitcher_hand_mix + self.list_pitcher_hand_count_mix +
                         self.list_batter_mix)
        list_mix_present = [c for c in list_mix_cols if c in X.columns]
        X_mix = X[list_mix_present].fillna(0).copy()
        X_cat = pd.get_dummies(X[self.list_categorical], columns=self.list_categorical, dtype=int)
        X_combined = pd.concat([X_num, X_mix, X_cat], axis=1).fillna(0)
        self.list_all_cols = sorted(X_combined.columns.tolist())
        return self
    def transform(self, X):
        print('Preparing features...')
        X = X.copy()
        # numeric
        X_num = X[self.list_numeric].copy()
        # all mix features (fill NaN with 0 for unseen pitchers/batters)
        list_mix_cols = (self.list_pitcher_mix + self.list_pitcher_count_mix +
                         self.list_pitcher_hand_mix + self.list_pitcher_hand_count_mix +
                         self.list_batter_mix)
        list_mix_present = [c for c in list_mix_cols if c in X.columns]
        X_mix = X[list_mix_present].fillna(0).copy()
        # categorical -> one-hot
        X_cat = pd.get_dummies(X[self.list_categorical], columns=self.list_categorical, dtype=int)
        # combine
        X_out = pd.concat([X_num, X_mix, X_cat], axis=1).fillna(0)
        # align columns to training set
        X_out = X_out.reindex(columns=self.list_all_cols, fill_value=0)
        return X_out

##### PreprocessingModel

A wrapper class that chains together all fitted transformers into a single callable pipeline. It takes a list of already-fit transformer instances and applies each one's `transform` method sequentially.

This class has no `fit` method by design — each individual transformer is fit separately on the training data beforehand. The `PreprocessingModel` is then used to transform validation and test splits (and future unseen data) through the exact same sequence of transformations, ensuring consistency. The full pipeline is serialized via `joblib.dump` so that at inference time, a single object can reproduce all preprocessing steps.

In [12]:
class PreprocessingModel:
    def __init__(self, list_cls_transformers):
        self.list_cls_transformers = list_cls_transformers
    def transform(self, X):
        for cls_transformer in self.list_cls_transformers:
            X = cls_transformer.transform(X)
        return X

#### Import Data from S3

In [13]:
%%time

df_train = pd.read_csv(f's3://{str_bucket}/02_data_split/train.csv')
df_valid = pd.read_csv(f's3://{str_bucket}/02_data_split/valid.csv')
df_test = pd.read_csv(f's3://{str_bucket}/02_data_split/test.csv')

print(f'Train: {df_train.shape}')
print(f'Valid: {df_valid.shape}')
print(f'Test:  {df_test.shape}')

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/fsspec/registry.py:301: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)
<timed exec>:1: DtypeWarning: Columns (29,30) have mixed types. Specify dtype option on import or set low_memory=False.
<timed exec>:2: DtypeWarning: Columns (29) have mixed types. Specify dtype option on import or set low_memory=False.


Train: (538294, 125)
Valid: (120479, 125)
Test:  (39545, 125)
CPU times: user 9.51 s, sys: 1.19 s, total: 10.7 s
Wall time: 27.6 s


<timed exec>:3: DtypeWarning: Columns (29) have mixed types. Specify dtype option on import or set low_memory=False.


#### Fit and Transform on Training Data

In [14]:
# situation features
cls_situation = SituationFeatures()
cls_situation.fit(df_train)
df_train = cls_situation.transform(df_train)

Situation features...
Situation features...


In [15]:
# lag features
cls_lag = LagFeatures()
cls_lag.fit(df_train)
df_train = cls_lag.transform(df_train)

Lag features...
Lag features...


In [16]:
# game context features (must come after SituationFeatures which creates score_diff)
cls_game_context = GameContextFeatures()
cls_game_context.fit(df_train)
df_train = cls_game_context.transform(df_train)

Game context features...
Game context features...


In [17]:
# pitcher stats
cls_pitcher = PitcherStats()
cls_pitcher.fit(df_train)
df_train = cls_pitcher.transform(df_train)

Computing pitcher stats...
Applying pitcher stats...


In [18]:
# pitcher handedness stats (must come after SituationFeatures which creates same_hand)
cls_pitcher_hand = PitcherHandednessStats()
cls_pitcher_hand.fit(df_train)
df_train = cls_pitcher_hand.transform(df_train)

Computing pitcher handedness stats...
Applying pitcher handedness stats...


In [19]:
# pitcher handedness count stats (must come after SituationFeatures which creates same_hand)
cls_pitcher_hand_count = PitcherHandednessCountStats()
cls_pitcher_hand_count.fit(df_train)
df_train = cls_pitcher_hand_count.transform(df_train)

Computing pitcher handedness count stats...
Applying pitcher handedness count stats...


In [20]:
# batter stats
cls_batter = BatterStats()
cls_batter.fit(df_train)
df_train = cls_batter.transform(df_train)

Computing batter stats...
Applying batter stats...


In [21]:
# count category
cls_count_cat = CountCategory()
cls_count_cat.fit(df_train)
df_train = cls_count_cat.transform(df_train)

Count category...
Count category...


In [ ]:
# prepare features
list_numeric = [
    'inning', 'top', 'outs', 'balls', 'strikes', 'fouls',
    'pcount_at_bat', 'pcount_pitcher', 'at_bat_num',
    'score_diff', 'total_runs', 'close_game', 'late_inning',
    'on_1b_flag', 'on_2b_flag', 'on_3b_flag', 'runners_on', 'bases_loaded',
    'same_hand', 'is_first_pitch',
    'b_height_inches',
    'pitcher_n_types',
]
list_categorical = ['p_throws', 'stand', 'prev_pitch_type', 'prev_pitch_type_2', 'count_category']

cls_prepare = PrepareFeatures(list_numeric=list_numeric, list_categorical=list_categorical)
cls_prepare.fit(df_train)
X_train = cls_prepare.transform(df_train)
y_train = df_train[str_target].copy()

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')

#### Build Preprocessing Model

In [23]:
list_cls_transformers = [
    cls_situation,
    cls_lag,
    cls_game_context,
    cls_pitcher,
    cls_pitcher_hand,
    cls_pitcher_hand_count,
    cls_batter,
    cls_count_cat,
    cls_prepare,
]
cls_model_preprocessing = PreprocessingModel(
    list_cls_transformers=list_cls_transformers,
)

#### Save Preprocessing Model Locally

In [24]:
str_filename = 'cls_model_preprocessing.joblib'
str_local_path = f'{str_dirname_output}/{str_filename}'
joblib.dump(cls_model_preprocessing, str_local_path)
print(f'Saved preprocessing model to {str_local_path}')

Saved preprocessing model to ./output/cls_model_preprocessing.joblib


#### Transform Valid and Test

In [25]:
X_valid = cls_model_preprocessing.transform(df_valid)
y_valid = df_valid[str_target].copy()

X_test = cls_model_preprocessing.transform(df_test)
y_test = df_test[str_target].copy()

print(f'X_valid: {X_valid.shape}, y_valid: {y_valid.shape}')
print(f'X_test:  {X_test.shape}, y_test:  {y_test.shape}')

Situation features...
Lag features...
Game context features...
Applying pitcher stats...
Applying pitcher handedness stats...
Applying pitcher handedness count stats...
Applying batter stats...
Count category...
Preparing features...
Situation features...
Lag features...
Game context features...
Applying pitcher stats...
Applying pitcher handedness stats...
Applying pitcher handedness count stats...
Applying batter stats...
Count category...
Preparing features...
X_valid: (120479, 90), y_valid: (120479,)
X_test:  (39545, 90), y_test:  (39545,)


In [26]:
# verify feature columns
print(f'Aligned feature count: {len(X_train.columns)}')
print('\nFeature columns:')
for col in sorted(X_train.columns):
    print(f'  {col}')

Aligned feature count: 90

Feature columns:
  at_bat_num
  b_height_inches
  balls
  batter_pct_CH
  batter_pct_CU
  batter_pct_FC
  batter_pct_FF
  batter_pct_FS
  batter_pct_FT
  batter_pct_SI
  batter_pct_SL
  batter_total_pitches
  close_game
  count_category_ahead
  count_category_behind
  count_category_even
  count_category_first_pitch
  count_category_full_count
  fouls
  inning
  is_first_pitch
  late_inning
  on_1b_flag
  on_2b_flag
  on_3b_flag
  outs
  p_throws_L
  p_throws_R
  pcount_at_bat
  pcount_pitcher
  pitcher_count_pct_CH
  pitcher_count_pct_CU
  pitcher_count_pct_FC
  pitcher_count_pct_FF
  pitcher_count_pct_FS
  pitcher_count_pct_FT
  pitcher_count_pct_SI
  pitcher_count_pct_SL
  pitcher_hand_count_pct_CH
  pitcher_hand_count_pct_CU
  pitcher_hand_count_pct_FC
  pitcher_hand_count_pct_FF
  pitcher_hand_count_pct_FS
  pitcher_hand_count_pct_FT
  pitcher_hand_count_pct_SI
  pitcher_hand_count_pct_SL
  pitcher_hand_pct_CH
  pitcher_hand_pct_CU
  pitcher_hand_pct_FC


#### Save to s3

In [27]:
# save feature matrices and targets to S3
for str_name, df_x, sr_y in [('train', X_train, y_train), ('valid', X_valid, y_valid), ('test', X_test, y_test)]:
    df_x.to_csv(f's3://{str_bucket}/{str_task}/X_{str_name}.csv', index=False)
    sr_y.to_csv(f's3://{str_bucket}/{str_task}/y_{str_name}.csv', index=False)
    print(f'Saved X_{str_name}.csv ({df_x.shape}) and y_{str_name}.csv ({sr_y.shape}) to S3')

Saved X_train.csv ((538294, 90)) and y_train.csv ((538294,)) to S3
Saved X_valid.csv ((120479, 90)) and y_valid.csv ((120479,)) to S3
Saved X_test.csv ((39545, 90)) and y_test.csv ((39545,)) to S3
